# Enregistrer la règle et son état

Rassembler les paramètres et les calculs dans un module.

Nous exécutons les cellules dans l’ordre, sur **CPU**. Aucun téléchargement de données ou de modèle n’est nécessaire.

Version de référence : **Python 3.11, PyTorch 2.10.0+cpu**. La cellule suivante indique la version réellement utilisée. Les sorties peuvent varier avec une autre version.

[Relire le chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#construire-un-module). Les deux exercices sont des textes à compléter : ils n’interrompent pas l’exécution intégrale. Leurs solutions restent repliées à la fin.

In [1]:
import platform
import torch

torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", platform.python_version())
print("PyTorch", torch.__version__)
print("Calcul", "CPU", "— threads", torch.get_num_threads())

Python 3.11.9
PyTorch 2.10.0+cpu
Calcul CPU — threads 1


## Nos repères

- **Module (module)** : Un objet qui regroupe calcul, paramètres et sous-couches.
- **Couche (layer)** : Une transformation précise du modèle.
- **Poids (weight)** : Un paramètre qui multiplie une contribution.
- **Tampon (buffer)** : Un état enregistré sans être un paramètre optimisé.

## Le calcul du cours

Avant de lancer cette cellule, annonçons les entrées et la sortie attendue. Après l’exécution, expliquons les nombres et les formes affichés. Le programme est identique à celui du chapitre.

In [2]:
import torch
from torch import nn

class Classifieur(nn.Module):
    def __init__(self):
        super().__init__()
        self.couches = nn.ModuleList([nn.Linear(3, 2, bias=False)])
        self.register_buffer("moyenne", torch.zeros(3))
    def forward(self, x):
        return self.couches[0](x - self.moyenne)

model = Classifieur()
with torch.no_grad():
    model.couches[0].weight.copy_(torch.tensor([[2., -1., .5], [-2., 1., -.5]]))
X = torch.tensor([[1., 2., 3.], [2., 0., 1.]])
scores = model(X)
print("scores", scores.tolist())
print("parametres", sum(p.numel() for p in model.parameters()))
print("buffer", model.moyenne.tolist())


scores [[1.5, -1.5], [4.5, -4.5]]
parametres 6
buffer [0.0, 0.0, 0.0]


## Nous vérifions notre compréhension

À dire en entretien : Je sépare les paramètres appris et les états fixes. Le module enregistre ses couches ; forward décrit comment leurs valeurs produisent les sorties.

## Exercice 1 — Conserver un état sans l’entraîner

Le décalage appartient au modèle mais n’est pas un paramètre appris. Complétons son enregistrement.

Remplaçons les points de suspension. Nous pouvons copier ce texte dans une nouvelle cellule de code.

```python
from torch import nn
class Centree(nn.Module):
    def __init__(self):
        super().__init__()
        self....("reference", torch.tensor([1.]))
        self.linear = nn.Linear(1, 1, bias=False)
    def forward(self, x):
        return self.linear(x - self.reference)
model = Centree()
print(list(model.state_dict()))
```

## Exercice 2 — Enregistrer plusieurs couches

Une collection de couches doit être connue du module. Complétons la collection et le parcours du calcul.

Remplaçons les points de suspension. Nous pouvons copier ce texte dans une nouvelle cellule de code.

```python
from torch import nn
class Petit(nn.Module):
    def __init__(self):
        super().__init__()
        self.couches = nn....([nn.Linear(2, 3), nn.Linear(3, 1)])
    def forward(self, x):
        for couche in self.couches:
            x = ...
        return x
model = Petit()
print(tuple(model(torch.ones(4, 2)).shape))
```

# Solutions

Nous cherchons d’abord, puis ouvrons la solution utile. Chaque programme ci-dessous est autonome. Le vérificateur du dépôt exécute également ces solutions, séparément des exercices à trous.

<details>
<summary>Afficher la solution de l’exercice 1 — Conserver un état sans l’entraîner</summary>

<pre><code>import torch
from torch import nn
class Centree(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer(&quot;reference&quot;, torch.tensor([1.]))
        self.linear = nn.Linear(1, 1, bias=False)
    def forward(self, x):
        return self.linear(x - self.reference)
model = Centree()
assert &quot;reference&quot; in model.state_dict()
assert &quot;reference&quot; not in dict(model.named_parameters())
print(list(model.state_dict()))</code></pre>
</details>

<details>
<summary>Afficher la solution de l’exercice 2 — Enregistrer plusieurs couches</summary>

<pre><code>import torch
from torch import nn
torch.manual_seed(17)
class Petit(nn.Module):
    def __init__(self):
        super().__init__()
        self.couches = nn.ModuleList([nn.Linear(2, 3), nn.Linear(3, 1)])
    def forward(self, x):
        for couche in self.couches:
            x = couche(x)
        return x
model = Petit()
sortie = model(torch.ones(4, 2))
assert tuple(sortie.shape) == (4, 1)
assert len(list(model.parameters())) == 4
print(tuple(sortie.shape))</code></pre>
</details>